# 02 - Data Understanding and Data Quality Assessment

## Objective

This notebook profiles the registered Online Retail MLTable before any cleaning or feature engineering is applied.

The purpose is to:
- validate the dataset structure;
- assess completeness and duplication;
- identify potentially invalid or business-significant transactions;
- understand the customer and transaction coverage;
- establish data-quality decisions that will inform downstream feature engineering and clustering.

No records are removed in this notebook unless explicitly justified.

In [3]:
# necessary libraries
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml.entities import Data
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import AccountKeyConfiguration
import mltable
from mltable import MLTableHeaders, MLTableFileEncoding

ml_client = MLClient.from_config(credential=DefaultAzureCredential())

# Load the registered Azure ML Data Asset so that analysis is based on the
# governed MLTable rather than a local file path.
retail_asset =ml_client.data.get(
    name="online-retail-mltable",
    version="1"
)

# Load the MLTable definition and materialise the dataset as a Pandas DataFrame
# for interactive profiling and exploratory analysis.
retail_table = mltable.load(retail_asset.path)
df = retail_table.to_pandas_dataframe()

Found the config file in: /config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


## 1. Validate dataset structure

Confirm that the registered data asset has the expected number of rows,
columns, field names and data types before performing any analysis.

In [2]:
# Confirm the overall size of the dataset.
df.shape

(541909, 8)

In [3]:
# Preview a small sample to understand the record structure and field values.
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [4]:
# Review column names, non-null counts and data types.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  string        
 1   StockCode    541909 non-null  string        
 2   Description  540455 non-null  string        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  string        
dtypes: datetime64[ns](1), float64(2), int64(1), string(4)
memory usage: 33.1 MB


In [5]:
# Confirm the available fields explicitly.
df.columns.tolist()

['InvoiceNo',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'UnitPrice',
 'CustomerID',
 'Country']

## 2. Assess missing values

Missing values can affect customer-level feature engineering, particularly
where CustomerID is required to aggregate transactions to individual customers.

In [7]:
# Count missing values in each column.
missing_counts = df.isna().sum()

import pandas as pd

# Calculate the percentage of missing values to understand their materiality.
missing_percentages = (
    df.isna().mean()
    .mul(100)
    .round(2)
)

missing_summary = (
    pd.DataFrame({
        "missing_count": missing_counts,
        "missing_percentage": missing_percentages
    })
    .sort_values("missing_percentage", ascending=False)
)

missing_summary

,missing_count,missing_percentage
CustomerID,135080,24.93
Description,1454,0.27
StockCode,0,0.00
InvoiceNo,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
Country,0,0.00


## 3. Assess duplicate transactions

Exact duplicate rows may represent duplicated source records or legitimate
repeated transaction lines. At this stage they are only quantified and are
not removed automatically.

In [8]:
# Count exact duplicate rows across all columns.
duplicate_count = df.duplicated().sum()

duplicate_count

5268

## 4. Identify cancelled transactions

In the source dataset, invoice numbers beginning with "C" indicate cancelled
transactions. These records may be relevant for understanding returns or
customer behaviour and therefore should not be removed without assessment.

In [9]:
# Identify invoices that begin with "C", which represent cancellations.
cancelled_mask = df["InvoiceNo"].astype("string").str.startswith("C", na=False)

cancelled_count = cancelled_mask.sum()

cancelled_count

9288

In [10]:
# Calculate the proportion of all rows associated with cancelled invoices.
cancelled_percentage = round(
    cancelled_count / len(df) * 100,
    2
)

cancelled_percentage

1.71

## 5. Assess non-positive quantities

Zero or negative quantities may represent returns, cancellations or other
non-standard transaction activity. These records require business
interpretation before deciding whether they should contribute to clustering
features.

In [11]:
# Count rows where quantity is zero or negative.
non_positive_quantity_count = (df["Quantity"] <= 0).sum()

non_positive_quantity_count

10624

## 6. Assess non-positive unit prices

Transactions with zero or negative prices may represent adjustments, free
items, data-quality issues or other exceptional activity.

In [12]:
# Count rows where UnitPrice is zero or negative.
non_positive_price_count = (df["UnitPrice"] <= 0).sum()

non_positive_price_count

2517

## 7. Understand customer coverage

Customer segmentation requires a reliable customer identifier. This section
measures how many identifiable customers are available for downstream
aggregation.

In [13]:
# Count distinct non-null customers represented in the dataset.
unique_customers = df["CustomerID"].nunique()

unique_customers

4372

## 8. Understand geographic coverage

Review the countries represented in the data because geographic distribution
may influence purchasing behaviour and later feature interpretation.

In [14]:
# Count the number of distinct countries represented.
unique_countries = df["Country"].nunique()

unique_countries

38

In [15]:
# Review transaction volume by country to identify dominant markets.
country_distribution = (
    df["Country"]
    .value_counts()
    .rename_axis("Country")
    .reset_index(name="transaction_rows")
)

country_distribution.head(10)

,Country,transaction_rows
0,United Kingdom,495478
1,Germany,9495
2,France,8557
3,EIRE,8196
4,Spain,2533
5,Netherlands,2371
6,Belgium,2069
7,Switzerland,2002
8,Portugal,1519
9,Australia,1259


## 9. Validate transaction period

Confirm the time span covered by the source dataset because recency and
customer tenure features depend directly on the available transaction window.

In [16]:
# Identify the earliest and latest transaction timestamps.
transaction_start = df["InvoiceDate"].min()
transaction_end = df["InvoiceDate"].max()

print("Transaction start:", transaction_start)
print("Transaction end:", transaction_end)

Transaction start: 2010-12-01 08:26:00
Transaction end: 2011-12-09 12:50:00


## 10. Data-quality findings and proposed treatment

The table below consolidates the main data-quality findings identified during
profiling. Proposed treatments are documented before cleaning is implemented
to ensure that transformation decisions are transparent and reproducible.

These treatments remain subject to validation during the detailed cleaning
and feature-engineering stage.

In [17]:
# Calculate specific data-quality measures that will be used in the summary table.

missing_customer_count = df["CustomerID"].isna().sum()
missing_customer_percentage = round(
    missing_customer_count / len(df) * 100,
    2
)

duplicate_percentage = round(
    duplicate_count / len(df) * 100,
    2
)

non_positive_quantity_percentage = round(
    non_positive_quantity_count / len(df) * 100,
    2
)

non_positive_price_percentage = round(
    non_positive_price_count / len(df) * 100,
    2
)

In [21]:
# Summarise the key data-quality findings and record the proposed treatment
# before implementing any cleaning logic.

data_quality_assessment = pd.DataFrame([
    {
        "Issue": "Missing CustomerID",
        "Observation": (
            f"{missing_customer_count:,} rows "
            f"({missing_customer_percentage}%) have no CustomerID."
        ),
        "Business interpretation": (
            "Transactions without a CustomerID cannot be reliably assigned "
            "to an individual customer."
        ),
        "Proposed treatment": (
            "Exclude these records from customer-level clustering."
        ),
        "Reason": (
            "Customer segmentation requires a stable customer identifier "
            "to aggregate transaction behaviour."
        )
    },
    {
        "Issue": "Exact duplicate rows",
        "Observation": (
            f"{duplicate_count:,} rows "
            f"({duplicate_percentage}%) are exact duplicates."
        ),
        "Business interpretation": (
            "Duplicate records may cause transaction value and purchase "
            "frequency to be overstated."
        ),
        "Proposed treatment": (
            "Investigate duplicate records before deciding whether to remove them."
        ),
        "Reason": (
            "Exact duplicates should not be removed automatically until it is "
            "confirmed that they are duplicate source records rather than "
            "legitimate repeated transaction lines."
        )
    },
    {
        "Issue": "Cancelled invoices",
        "Observation": (
            f"{cancelled_count:,} rows "
            f"({cancelled_percentage}%) belong to invoices beginning with 'C'."
        ),
        "Business interpretation": (
            "These transactions represent cancellations and may reflect "
            "returns or reversed purchases."
        ),
        "Proposed treatment": (
            "Exclude cancellations from positive purchase calculations, "
            "but assess whether cancellation behaviour should be retained "
            "as a separate customer feature."
        ),
        "Reason": (
            "Treating cancelled transactions as normal purchases could distort "
            "Recency, Frequency and Monetary features."
        )
    },
    {
        "Issue": "Non-positive quantity",
        "Observation": (
            f"{non_positive_quantity_count:,} rows "
            f"({non_positive_quantity_percentage}%) have Quantity <= 0."
        ),
        "Business interpretation": (
            "Negative quantities are likely associated with returns or "
            "cancelled transactions."
        ),
        "Proposed treatment": (
            "Investigate their relationship with cancelled invoices before "
            "excluding them from purchase-based features."
        ),
        "Reason": (
            "Negative quantities should not contribute to standard purchase "
            "volume without understanding their business meaning."
        )
    },
    {
        "Issue": "Non-positive UnitPrice",
        "Observation": (
            f"{non_positive_price_count:,} rows "
            f"({non_positive_price_percentage}%) have UnitPrice <= 0."
        ),
        "Business interpretation": (
            "These may represent free items, adjustments, errors or other "
            "non-standard transactions."
        ),
        "Proposed treatment": (
            "Investigate the affected records before deciding whether they "
            "should be excluded."
        ),
        "Reason": (
            "Zero or negative prices could distort customer monetary-value "
            "features."
        )
    }
])


In [20]:
# Display the assessment table with wrapped text so that long explanations
# remain readable within the notebook.

display(
    data_quality_assessment.style
    .set_properties(
        **{
            "white-space": "normal",
            "text-align": "left",
            "vertical-align": "top"
        }
    )
)

,Issue,Observation,Business interpretation,Proposed treatment,Reason
0,Missing CustomerID,"135,080 rows (24.93%) have no CustomerID.",Transactions without a CustomerID cannot be reliably assigned to an individual customer.,Exclude these records from customer-level clustering.,Customer segmentation requires a stable customer identifier to aggregate transaction behaviour.
1,Exact duplicate rows,"5,268 rows (0.97%) are exact duplicates.",Duplicate records may cause transaction value and purchase frequency to be overstated.,Investigate duplicate records before deciding whether to remove them.,Exact duplicates should not be removed automatically until it is confirmed that they are duplicate source records rather than legitimate repeated transaction lines.
2,Cancelled invoices,"9,288 rows (1.71%) belong to invoices beginning with 'C'.",These transactions represent cancellations and may reflect returns or reversed purchases.,"Exclude cancellations from positive purchase calculations, but assess whether cancellation behaviour should be retained as a separate customer feature.","Treating cancelled transactions as normal purchases could distort Recency, Frequency and Monetary features."
3,Non-positive quantity,"10,624 rows (1.96%) have Quantity <= 0.",Negative quantities are likely associated with returns or cancelled transactions.,Investigate their relationship with cancelled invoices before excluding them from purchase-based features.,Negative quantities should not contribute to standard purchase volume without understanding their business meaning.
4,Non-positive UnitPrice,"2,517 rows (0.46%) have UnitPrice <= 0.","These may represent free items, adjustments, errors or other non-standard transactions.",Investigate the affected records before deciding whether they should be excluded.,Zero or negative prices could distort customer monetary-value features.


## 11. Automated Data Profiling

An automated profiling report is generated to complement the targeted
data-quality checks performed above.

The report provides an additional view of:

- column distributions;
- missing values;
- duplicate records;
- unique and high-cardinality values;
- descriptive statistics;
- potential data-quality warnings.

Automated profiling is used as a diagnostic aid only. Business-specific
treatment decisions continue to be based on the targeted analysis documented
in this notebook.

In [5]:
%pip install -U ydata-profiling


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
%pip install -U ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 21.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]3 [ipywidgets]

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
from ydata_profiling import ProfileReport

# Generate an automated profile of the full dataset.
# minimal=True reduces expensive calculations such as detailed correlations,
# which is appropriate for the initial profiling of a large dataset.

profile = ProfileReport(
    df,
    title="Online Retail Data Profiling Report",
    minimal=True
)

# Display the interactive profiling report in the notebook.
profile.to_notebook_iframe()

Render HTML: 100%|██████████| 1/1 [00:00<00:00,  4.93it/s]


In [9]:
# saving this

import os

reports_folder = "../reports"
os.makedirs(reports_folder, exist_ok=True)

# Save the profiling report as a standalone HTML file.
profile.to_file(
    "../reports/online_retail_data_profile.html"
)

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 15.99it/s]
